# I5 — Three-seed heads for each S/B336/L336 backbone

Select T4 GPU, enable the `KAGGLE_API_TOKEN` Colab secret, authorize Drive and run all cells. Original F0/I1/I2 paths are prefilled. Keep their complete feature caches. I3 fine-tuning and I4 TTA are not used.

Preserve the original seed-2026 B336/L336 heads. Train only four new heads: B336 seeds 42/123 and L336 seeds 42/123. Each uses its original saved 3500 training-feature rows from 700 images; no new B/L extraction. The original three S224 heads remain unchanged. S224 original predictions are replayed to verify the source ensemble.

**Fixed three epochs for new heads, batch64, AdamW lr/decay .001. Validation is diagnostic only; no early stopping, best-epoch selection, seed selection or export gate based on validation improvement.** Three epochs matches the training duration of the saved original B/L checkpoints. Normalization is fixed from original training features.

The sole candidate averages seeds 42/123/2026 within each backbone, then averages the three backbone branches: all nine heads have weight 1/9. No TTA or backbone fine-tuning. The current 0.97959 CSV must reproduce before training. New outputs go under `student_outputs_I5_NineHead/<uuid>/`, including copies of the five preserved source head checkpoints and four new ones. No Kaggle submission happens automatically.

In [ ]:
%pip -q install scikit-learn pandas pillow kagglehub

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
STORAGE_ROOT=Path('/content/drive/MyDrive/Car-Classification-Benchmark')
F0_RUN_DIR=STORAGE_ROOT/'student_outputs_F0_Ensemble/20261108T091250Z_4ed04fc8'
I1_RUN_DIR=STORAGE_ROOT/'student_outputs_I1_ViTB336/1402790cb5a0'
I2_RUN_DIR=STORAGE_ROOT/'student_outputs_I2_ViTL336/6dcb19f3d7e4'
for folder in (F0_RUN_DIR,I1_RUN_DIR,I2_RUN_DIR):
    assert folder.is_dir(), f'Saved source folder missing: {folder}'
DOWNLOAD_FROM_KAGGLE=True
DATA_ROOT=Path('/content/car_data')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN']=userdata.get('KAGGLE_API_TOKEN')
    DATA_ROOT=Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))
for split in ('train','val','test'):
    assert (DATA_ROOT/split).is_dir(), f'Missing dataset split: {DATA_ROOT/split}'
print('Data:',DATA_ROOT)


In [ ]:
"""I5: three seeds per S/B336/L336 branch; fixed-head training on original caches."""
from pathlib import Path
import gc, hashlib, json, random, subprocess, time, uuid
from urllib.error import HTTPError
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image

import shutil
from torch.utils.data import TensorDataset


HEAD_RECIPE={'epochs':3,'batch_size':64,'learning_rate':.001,'weight_decay':.001,
             'optimizer':'AdamW','selection':'final epoch; validation diagnostic only'}

def full_feature_cache(source,storage_root,data_root,target,ids):
    meta=source['feature_metadata'];dim=len(source['feature_mean'])
    signature=hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()
    assert signature==source['feature_signature']
    paths=list(Path(storage_root).glob('student_outputs_I*/cache_336/*_'+signature+'.pt'))
    assert len(paths)==1,'Restore the original feature cache: '+signature
    p=torch.load(paths[0],map_location='cpu',weights_only=True)
    assert p['metadata']==meta and meta['test_ids']==ids and meta['image_size']==336
    assert meta['feature_dim']==dim and meta['seed_aug']==123 and meta['n_aug']==4
    assert meta['include_original'] is True and meta['layout']=='view-major'
    training=datasets.ImageFolder(Path(data_root)/'train')
    assert training.class_to_idx==source['class_to_idx'] and len(training)==700
    for key,n in [('X_train',3500),('X_val',len(target)),('X_test',len(ids))]:
        assert p[key].shape==(n,dim) and p[key].dtype==torch.float32 and torch.isfinite(p[key]).all()
    assert torch.equal(p['y_train'],torch.tensor(training.targets).repeat(5))
    assert np.array_equal(p['y_val'].numpy(),target)
    assert torch.allclose(p['X_train'].mean(0),source['feature_mean'],rtol=1e-5,atol=1e-6)
    assert torch.allclose(p['X_train'].std(0,unbiased=False).clamp_min(1e-6),source['feature_std'],rtol=1e-5,atol=1e-6)
    return p,paths[0]

@torch.inference_mode()
def head_probabilities(source,features,device):
    dim=len(source['feature_mean']);head=build_head(dim).to(device)
    head.load_state_dict(source['head_state_dict']);head.eval()
    result=[]
    for key in ('X_val','X_test'):
        x=(features[key]-source['feature_mean'])/source['feature_std']
        p=head(x.to(device)).softmax(1).cpu()
        assert p.shape==(len(x),20) and torch.isfinite(p).all()
        result.append(p)
    return result

def train_seed(seed,source,features,output,device):
    assert seed in (42,123),'Preserve the original 2026 head without retraining'
    assert source['best_epoch']==3,'I5 fixed duration matches the original three-epoch B/L checkpoints'
    assert source['training_config']=={'epochs':35,'batch_size':64,'learning_rate':.001,'weight_decay':.001,'patience':7,'optimizer':'AdamW'}
    mean,std=source['feature_mean'],source['feature_std']
    set_seed(seed);head=build_head(len(mean)).to(device)
    loader=DataLoader(TensorDataset((features['X_train']-mean)/std,features['y_train']),
        batch_size=64,shuffle=True,num_workers=0,generator=torch.Generator().manual_seed(seed))
    optimizer=torch.optim.AdamW(head.parameters(),lr=.001,weight_decay=.001)
    history=[]
    for epoch in range(1,HEAD_RECIPE['epochs']+1):
        head.train();loss_sum=0.
        for x,y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss=nn.functional.cross_entropy(head(x.to(device)),y.to(device));assert torch.isfinite(loss)
            loss.backward();optimizer.step();loss_sum+=loss.item()*len(y)
        head.eval()
        with torch.inference_mode():pv=head(((features['X_val']-mean)/std).to(device)).softmax(1).cpu()
        score=metrics(features['y_val'].numpy(),pv)
        history.append({'epoch':epoch,'train_loss':loss_sum/len(features['X_train']),**score})
        print('I5_HEAD',source['backbone'],'SEED',seed,'EPOCH',epoch,score,flush=True)
    weights={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
    result={**source,'head_state_dict':weights,'seed_head':seed,'selected_epoch':3,
        'metrics':score,'training_config':HEAD_RECIPE,'origin':'I5 new head; final epoch; no validation selection'}
    for key in ('best_epoch','best_accuracy'):result.pop(key,None)
    name=f"I5_{source['backbone']}_336_head{seed}"
    path=Path(output)/(name+'.pt')
    with path.open('xb') as stream:torch.save(result,stream)
    pd.DataFrame(history).to_csv(Path(output)/(name+'_history.csv'),index=False)
    pv,pt=head_probabilities(result,features,device)
    return pv,pt,path,result

def nine_head_blend(small,base,large):
    assert small.ndim==base.ndim==large.ndim==3
    assert small.shape==base.shape==large.shape and small.shape[0]==3 and small.shape[2]==20
    for p in (small,base,large):
        assert torch.isfinite(p).all() and (p>=0).all() and (p<=1).all()
        assert torch.allclose(p.sum(-1),torch.ones_like(p.sum(-1)),atol=1e-5)
    return equal_blend(small.mean(0),base.mean(0),large.mean(0))

def run_i5(data_root,f0_run,i1_run,i2_run,storage_root):
    assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before running I5.'
    device='cuda';set_seed(2026)
    mapping,val_paths,target,test_paths,ids=load_data(data_root)
    f0_run,i1_run,i2_run=map(Path,(f0_run,i1_run,i2_run))
    f0=json.loads((f0_run/'ensemble_summary.json').read_text())
    ref=json.loads((i2_run/'summary.json').read_text())
    assert f0['head_seeds']==SEEDS and f0['hub_ref']==HUB_REF and f0['augmentation_seed']==123
    assert np.allclose(f0['weights'],[1/3]*3)
    assert np.allclose(ref['effective_head_weights'],[1/9,1/9,1/9,1/3,1/3])
    expected={Path(p).name:h for p,h in ref['source_checkpoint_sha256'].items()}
    rows={r['seed']:r for r in f0['individual_validation']}
    small_paths=[f0_run/Path(rows[s]['checkpoint']).name for s in SEEDS]
    base_path=i1_run/'I1_dinov2_vitb14_336_p3_e4_aug123_head2026.pt'
    large_path=i2_run/Path(ref['new_checkpoint']).name
    expected[large_path.name]=ref['new_checkpoint_sha256']
    paths=small_paths+[base_path,large_path]
    original={str(p):sha(p) for p in paths}
    for p in paths: assert original[str(p)]==expected[p.name], 'Source checkpoint changed: '+str(p)
    small=[checked_checkpoint(p,'dinov2_vits14',384,s,mapping) for p,s in zip(small_paths,SEEDS)]
    base=checked_checkpoint(base_path,'dinov2_vitb14',768,2026,mapping)
    large=torch.load(large_path,map_location='cpu',weights_only=True)
    assert large['backbone']=='dinov2_vitl14' and large['resolution']==336 and large['class_to_idx']==mapping
    assert base['resolution']==336 and base['feature_metadata']['image_size']==336
    assert large['hub_ref']==HUB_REF and large['feature_metadata']['manifest']==base['feature_metadata']['manifest']
    assert large['feature_mean'].shape==large['feature_std'].shape==(1024,)
    assert torch.isfinite(large['feature_mean']).all() and torch.isfinite(large['feature_std']).all() and (large['feature_std']>0).all()
    validate_source_data(base,data_root,mapping,ids)
    vb,tb=cached_probabilities(base,storage_root,target,ids)
    vl,tl=cached_probabilities(large,storage_root,target,ids)
    reproduce(metrics(target,vb),base['metrics'],'B336 cache')
    reproduce(metrics(target,vl),large['metrics'],'L336 cache')
    evaluation,augmentation=transforms336()
    eval224=transforms.Compose([transforms.Resize(256,interpolation=InterpolationMode.BICUBIC),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    encoder=load_official('dinov2_vits14',f0['weights_sha256'],384,device)
    vs=infer(encoder,small,val_paths,eval224,device,8)
    ts=infer(encoder,small,test_paths,eval224,device,8)
    for i,seed in enumerate(SEEDS): reproduce(metrics(target,vs[i]),rows[seed],f'F0 seed{seed}')
    reproduce(metrics(target,vs.mean(0)),f0['ensemble_validation'],'F0 ensemble')
    current_val=equal_blend(vs.mean(0),vb,vl);current_test=equal_blend(ts.mean(0),tb,tl)
    reproduce(metrics(target,current_val),ref['validation']['new_equal_S_B_L_blend'],'Current 0.97959 ensemble')
    control_path=i2_run/'F0_B336_L336_equal_backbone_submission.csv'
    control=pd.read_csv(control_path,dtype={'image_id':str})
    assert control.image_id.tolist()==ids and np.array_equal(control.label.to_numpy(),current_test.argmax(1).numpy()), 'Winning submission did not reproduce'
    del encoder;gc.collect();torch.cuda.empty_cache()
    root=Path(storage_root)/'student_outputs_I5_NineHead'
    output=root/uuid.uuid4().hex[:12];output.mkdir(parents=True,exist_ok=False)
    print('OUTPUT',output,flush=True)
    branches={};new_paths=[];new_metrics=[];cache_sources={}
    for branch,source,original_v,original_t in [('B336',base,vb,tb),('L336',large,vl,tl)]:
        assert source['best_epoch']==3
        features,cache=full_feature_cache(source,storage_root,data_root,target,ids)
        cache_sources[branch]={'path':str(cache),'signature':source['feature_signature']}
        replay_v,replay_t=head_probabilities(source,features,device)
        assert torch.allclose(replay_v,original_v,atol=2e-5,rtol=2e-4)
        assert torch.allclose(replay_t,original_t,atol=2e-5,rtol=2e-4)
        val_parts=[];test_parts=[]
        for seed in (42,123):
            pv,pt,path,checkpoint=train_seed(seed,source,features,output,device)
            val_parts.append(pv);test_parts.append(pt);new_paths.append(path)
            new_metrics.append({'branch':branch,'seed':seed,**checkpoint['metrics']})
        val_parts.append(original_v);test_parts.append(original_t)
        branches[branch]=(torch.stack(val_parts),torch.stack(test_parts))
        del features;gc.collect()
    candidate_v=nine_head_blend(vs,branches['B336'][0],branches['L336'][0])
    candidate_t=nine_head_blend(ts,branches['B336'][1],branches['L336'][1])
    for path in paths:assert sha(path)==original[str(path)],'Original checkpoint modified'
    for path in paths:
        destination=output/path.name
        with destination.open('xb') as stream, path.open('rb') as source_stream:shutil.copyfileobj(source_stream,stream)
        assert sha(destination)==original[str(path)]
    export(output/'CurrentBest_097959_control.csv',ids,current_test)
    export(output/'I5_S_B336_L336_three_seed_submission.csv',ids,candidate_t)
    changed=current_test.argmax(1)!=candidate_t.argmax(1)
    pd.DataFrame({'image_id':ids,'original_label':current_test.argmax(1).numpy(),
        'candidate_label':candidate_t.argmax(1).numpy(),'changed':changed.numpy()}).to_csv(output/'test_prediction_changes.csv',index=False)
    summary={'experiment':'I5 three seeds per S/B336/L336 backbone','head_seeds':SEEDS,
        'branch_weights':[1/3]*3,'effective_head_weights':[1/9]*9,'head_recipe':HEAD_RECIPE,
        'original_validation':metrics(target,current_val),'candidate_validation':metrics(target,candidate_v),
        'new_heads_validation':new_metrics,'test_changed':int(changed.sum()),'cache_sources':cache_sources,
        'source_checkpoint_sha256':original,'new_checkpoint_sha256':{str(p):sha(p) for p in new_paths},
        'reference_submission_sha256':sha(control_path),'historical_best_kaggle':.97959,
        'source_predictions_reproduced':True,'validation_selection':False,'TTA':False,
        'backbones_frozen':True,'test_labels_used':False,'submitted':False}
    np.savez_compressed(output/'probabilities.npz',validation_original=current_val.numpy(),validation_candidate=candidate_v.numpy(),
        test_original=current_test.numpy(),test_candidate=candidate_t.numpy(),targets=target,
        B336_validation_heads=branches['B336'][0].numpy(),B336_test_heads=branches['B336'][1].numpy(),
        L336_validation_heads=branches['L336'][0].numpy(),L336_test_heads=branches['L336'][1].numpy())
    (output/'summary.json').write_text(json.dumps(summary,indent=2))
    print('COMPLETE',output,json.dumps(summary),flush=True)
    return output,summary

HUB_REF='7764ea0f912e53c92e82eb78a2a1631e92725fc8'
SEEDS=[42,123,2026]
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def weight_digest(model):
    h=hashlib.sha256()
    for name,value in sorted(model.state_dict().items()):
        h.update(name.encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def build_head(dim):
    return nn.Sequential(nn.Linear(dim,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,20))

class Images(torch.utils.data.Dataset):
    def __init__(self,paths,transform):self.paths,self.transform=paths,transform
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as image:return self.transform(image.convert('RGB'))

def load_data(root):
    root=Path(root)
    train,val=[datasets.ImageFolder(root/name) for name in ('train','val')]
    assert len(train)==700 and len(val)==200 and len(train.classes)==20
    assert train.class_to_idx==val.class_to_idx
    assert all(name.startswith(f'{i:03d}_') for i,name in enumerate(train.classes))
    sample=pd.read_csv(root/'sample_submission.csv',dtype={'image_id':str})
    assert {'image_id','label'}<=set(sample.columns) and sample.image_id.is_unique and sample.image_id.notna().all()
    files=sorted(p for p in (root/'test').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
    by_id={p.stem:p for p in files};ids=sample.image_id.tolist()
    assert len(by_id)==len(files) and set(by_id)==set(ids)
    return train.class_to_idx,[Path(p) for p,_ in val.samples],np.asarray(val.targets),[by_id[i] for i in ids],ids

def checked_checkpoint(path,model,dim,seed,mapping):
    p=torch.load(path,map_location='cpu',weights_only=True)
    assert p['backbone']==model and p['seed_aug']==123 and p['seed_head']==seed
    assert p['class_to_idx']==mapping
    for key in ('feature_mean','feature_std'):
        assert p[key].shape==(dim,) and torch.isfinite(p[key]).all()
    assert (p['feature_std']>0).all()
    expected={'hidden_dim':128,'use_bn':True,'dropout':.3}
    if dim==768:expected={'input_dim':768,**expected,'num_classes':20}
    assert p['head_config']==expected,'Checkpoint is not the requested E4 architecture'
    build_head(dim).load_state_dict(p['head_state_dict'],strict=True)
    return p

@torch.inference_mode()
def infer(backbone,checkpoints,paths,transform,device,batch_size=8):
    backbone.eval();assert not any(p.requires_grad for p in backbone.parameters())
    heads=[]
    for c in checkpoints:
        h=build_head(len(c['feature_mean'])).to(device)
        h.load_state_dict(c['head_state_dict'],strict=True);h.eval()
        heads.append((h,c['feature_mean'].to(device),c['feature_std'].to(device)))
    results=[[] for _ in heads]
    loader=DataLoader(Images(paths,transform),batch_size=batch_size,shuffle=False,num_workers=0)
    for index,images in enumerate(loader):
        features=backbone(images.to(device)).float()
        for j,(head,mean,std) in enumerate(heads):
            assert features.shape==(len(images),len(mean))
            p=head((features-mean)/std).softmax(1)
            assert p.shape==(len(images),20) and torch.isfinite(p).all()
            results[j].append(p.cpu())
        if index%40==0:print('PREDICT_BATCH',index+1,'/',len(loader),flush=True)
    return torch.stack([torch.cat(parts) for parts in results])

def metrics(target,probabilities):
    from sklearn.metrics import accuracy_score,f1_score
    pred=probabilities.argmax(1).numpy()
    return {'accuracy':float(accuracy_score(target,pred)),
        'macro_f1':float(f1_score(target,pred,labels=list(range(20)),average='macro',zero_division=0)),
        'correct':int((pred==target).sum()),'count':len(target)}

def reproduce(measured,expected,name):
    assert measured['correct']==expected['correct'] and abs(measured['accuracy']-expected['accuracy'])<1e-12, (
        f'{name} starting accuracy did not reproduce; diagnose checkpoint/source/preprocessing before exporting')

def export(path,ids,probabilities):
    pred=probabilities.argmax(1).numpy().astype(int)
    assert len(pred)==len(ids)==len(set(ids)) and ((pred>=0)&(pred<20)).all()
    pd.DataFrame({'image_id':list(ids),'label':pred}).to_csv(path,index=False,mode='x')

def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)

def transforms336():
    tail=[transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])]
    evaluation=transforms.Compose([transforms.Resize(384,interpolation=InterpolationMode.BICUBIC),transforms.CenterCrop(336),*tail])
    training=transforms.Compose([transforms.RandomResizedCrop(336,scale=(.8,1.),ratio=(.9,1.1),interpolation=InterpolationMode.BICUBIC),transforms.RandomHorizontalFlip(.5),*tail])
    return evaluation,training

def validate_source_data(base,data_root,mapping,ids):
    meta=base['feature_metadata']
    assert meta.get('backbone',meta.get('model'))=='dinov2_vitb14'
    assert base['hub_ref']==HUB_REF and meta['class_to_idx']==mapping and meta['test_ids']==ids
    assert meta['feature_dim']==768 and meta['seed_aug']==123 and meta['n_aug']==4 and meta['include_original'] is True
    root=Path(data_root).resolve()
    for split in meta['manifest']:
        for entry in (split['files'] if isinstance(split,dict) else split):
            path=(root/entry['path']).resolve()
            assert path.is_relative_to(root) and path.is_file() and sha(path)==entry['sha256'],'Data differs from original I0 run'

def load_official(name,expected,dim,device):
    try:
        model=torch.hub.load(f'facebookresearch/dinov2:{HUB_REF}',name,pretrained=True,trust_repo=True,skip_validation=True)
    except HTTPError as exc:
        if exc.code not in (502,503,504):raise
        root=Path(torch.hub.get_dir())/('dinov2_git_'+HUB_REF[:12])
        if not root.exists():
            subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/facebookresearch/dinov2.git',str(root)],check=True)
        subprocess.run(['git','-C',str(root),'checkout','--detach',HUB_REF],check=True)
        assert subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()==HUB_REF
        model=torch.hub.load(str(root),name,source='local',pretrained=True)
    model.requires_grad_(False).eval()
    assert model.embed_dim==dim
    if expected is not None:assert weight_digest(model)==expected,'Backbone does not match original source weights'
    return model.to(device)

@torch.inference_mode()
def cached_probabilities(checkpoint,storage_root,target,ids):
    signature=checkpoint['feature_signature']
    assert signature==hashlib.sha256(json.dumps(checkpoint['feature_metadata'],sort_keys=True).encode()).hexdigest()
    matches=list(Path(storage_root).glob('student_outputs_I*/cache_336/*_'+signature+'.pt'))
    assert len(matches)==1, 'Required original feature cache missing/ambiguous: '+signature+'; restore it from the original run, do not re-extract L336.'
    cached=torch.load(matches[0],map_location='cpu',weights_only=True)
    assert cached['metadata']==checkpoint['feature_metadata']
    assert cached['metadata']['test_ids']==ids and cached['metadata']['image_size']==336
    assert np.array_equal(cached['y_val'].numpy(),target)
    head=build_head(len(checkpoint['feature_mean']));head.load_state_dict(checkpoint['head_state_dict']);head.eval()
    result=[]
    for key,n in [('X_val',len(target)),('X_test',len(ids))]:
        x=cached[key]
        assert x.shape==(n,len(checkpoint['feature_mean'])) and x.dtype==torch.float32 and torch.isfinite(x).all()
        result.append(head((x-checkpoint['feature_mean'])/checkpoint['feature_std']).softmax(1))
    return result

def equal_blend(small,base,large):
    assert small.shape==base.shape==large.shape and small.ndim==2 and small.shape[1]==20
    for p in (small,base,large):
        assert torch.isfinite(p).all() and (p>=0).all() and (p<=1).all()
        assert torch.allclose(p.sum(1),torch.ones(len(p)),atol=1e-5)
    return (small+base+large)/3



In [ ]:
output,summary=run_i5(DATA_ROOT,F0_RUN_DIR,I1_RUN_DIR,I2_RUN_DIR,STORAGE_ROOT)
from IPython.display import display
display(pd.DataFrame([{'model':name,**summary[key]} for name,key in [('original','original_validation'),('nine-head candidate','candidate_validation')]]))
print('Changed test predictions:',summary['test_changed'])
print('New candidate (not submitted):',output/'I5_S_B336_L336_three_seed_submission.csv')
print('Preserve the complete output folder:',output)